**Students:** Igor Kochanski 23358459, Emily Domini 23362235, Nikolay Stoiakine 23380969

**Run Status:** The code executes to the end without an error.

**Development Baseline:** https://github.com/kerrgarr/SemanticSegmentationCityscapes

--- ---

# U-Net Semantic Image Segmentation on Cityscapes

CS4287 - Neural Computing   
Assignment 1: Convolutional Neural Networks (CNNs)

# Introduction

Notebook implements a U-Net-based semantic segmentation model trained and evaluated on the Cityscapes dataset.

Sections:

1. [Dataset Preparation](#dataset-preparation)
2. [Exploratory Dataset Analysis](#exploratory-dataset-analysis)
3. [Model and Training](#model-and-training)
4. [Cross-Fold-Validation](#cross-fold-validation)

# Requirements

Requires Python 3.10+

Create virtual environment and activate it
```bash
# Linux/MacOS
python3 -m venv .venv
source .venv/bin/activate

# Windows
python -m venv .venv
.venv\Scripts\activate
```

Install requirements *(run cell below to generate `requirements.txt`)*
```bash
pip install -r requirements.txt
```

In [ ]:
%%writefile requirements.txt
nbstripout
jupyter
ipykernel
ipywidgets
tqdm
pandas
numpy
torch
torchinfo
torchvision
segmentation-models-pytorch
matplotlib
seaborn
imageio
cityscapesscripts

Download Cityscapes datasets: https://www.cityscapes-dataset.com/downloads

Datasets used: gtFine_trainvaltest.zip (241MB), leftmg8bit_trainvaltest.zip (11GB)

Unzip both files in the data/ directory

Your data/ folder structure should look like:

data/   
├──gtFine_trainvaltest/   
└──leftImg8bit_trainvaltest/    

In [ ]:
import csv
import json
import os
import shutil
import time
from pathlib import Path
from collections import Counter

from PIL import Image

import numpy as np
import pandas as pd
from tqdm import tqdm

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from torchinfo import summary
from segmentation_models_pytorch.losses import LovaszLoss

from cityscapesscripts.helpers.labels import labels

In [ ]:
# project dataset
DATASET_DIR = './data/dataset/'
# Downloaded unzipped Cityscapes datasets
LABELS_DIR = './data/gtFine_trainvaltest/gtFine/'
IMAGES_DIR = './data/leftImg8bit_trainvaltest/leftImg8bit/'

NUM_CLASSES = 19
IGNORE_INDEX = 255

FIGURE_DIR = "./eda/figures"
TABLES_DIR = "./eda/tables"
EDA_CLASS_DIST = "./eda/tables/split_class_distribution.csv"

SPLITS = ["train", "val", "test"]

MODELS_DIR = "./models" # save model

Helper functions

In [ ]:
# sort the 19 evaluated classes by their trainId (0-18) and ignore the rest
TRAIN_LABELS = sorted(
    (label for label in labels
     if not label.ignoreInEval and 0 <= label.trainId < 255),
    key=lambda label: label.trainId
)

CLASS_NAMES = [label.name for label in TRAIN_LABELS]
NUM_CLASSES = len(CLASS_NAMES)  # 19

# map from original label ID (0-33) to train ID (0-18, 255)
LABEL_TO_TRAIN = {
    label.id: label.trainId if label.trainId >= 0 else IGNORE_INDEX
    for label in labels
    if label.id >= 0
}

# lookup table indexed by original label ID (0-255)
ID_LOOKUP = np.full(256, IGNORE_INDEX, dtype=np.uint8)
for label_id, train_id in LABEL_TO_TRAIN.items():
    if label_id < 256:
        ID_LOOKUP[label_id] = train_id

# collect the colours of the 19 evaluated classes
CLASS_COLOURS = np.array(
    [label.color for label in TRAIN_LABELS],
    dtype=np.uint8
)  # shape (19, 3), row i = colour of train ID i

# map the class names to their corresponding colours as hex strings for plotting
CLASS_COLOURS_HEX = {
    label.name: "#{:02x}{:02x}{:02x}".format(*label.color)
    for label in TRAIN_LABELS
}

Hyperparameters

In [ ]:
VERSION = "v2.0.6" # model version for saving

# spatial config (for training only)
NATIVE_SIZE = (1024, 2048) # original resolution
CROP_SIZE = (512, 1024)    # random crop size | None = no crops
CROP_SCALE = (0.5, 2.0)    # random scale factor range for crops
TRAIN_SIZE = None          # resize if no crops | None = native size
FLIP = True                # random horizontal flip

INPUT_SIZE = CROP_SIZE or TRAIN_SIZE or NATIVE_SIZE # training samples size

TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 2
NUM_WORKERS = 8

# model
BASE = 32
# criterion
CE_COEFF = 1.0 # weight cross entropy loss
LOVASZ_COEFF = 0.5 # weight lovasz loss
WEIGHT_EXPONENT = 0.5 # class imbalance weights: 0 = no weights, 0.5 = sqrt
# optimizer
MAX_LEARNING_RATE = 1e-4 # start at this rate
MIN_LEARNING_RATE = 1e-6 # slowly reduce to this rate
WEIGHT_DECAY = 1e-4
# training
EPOCHS = 160

In [ ]:
# EDA
FIGURE_DIR = Path(FIGURE_DIR)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR = Path(TABLES_DIR)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
# MODELS
VERSION_SIZE = f"{VERSION}-{INPUT_SIZE[0]}-{INPUT_SIZE[1]}"
MODEL_DIR = Path(MODELS_DIR)/VERSION_SIZE
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL = MODEL_DIR/f"model-{VERSION_SIZE}.pt"
MODEL_CHECKPOINT = MODEL_DIR/f"checkpoint-{VERSION_SIZE}.pt"
MODEL_HISTORY = MODEL_DIR/"history.csv"
MODEL_GIF = MODEL_DIR/"training-gif"
MODEL_EVAL = MODEL_DIR/"evaluation.txt"
MODEL_LOSS = MODEL_DIR/"loss.png"
MODEL_CLASSES = MODEL_DIR/"classes.png"

# Dataset Preparation

In [ ]:
# See available cities in the dataset
for folder in os.listdir(IMAGES_DIR):
    cities = []
    for city in os.listdir(os.path.join(IMAGES_DIR, folder)):
        cities.append(city)
    print(f"{folder}: {cities}")

In [ ]:
# See how many images are in each of the original city folders
for folder in os.listdir(IMAGES_DIR):
    print(folder)
    for city in os.listdir(os.path.join(IMAGES_DIR, folder)):
        path = os.path.join(IMAGES_DIR, folder, city)
        print(f"  {city}: {len(os.listdir(path))}")

**NOTE** - The official Cityscapes **test** split does not contian image masks, only blanks. The test split is used by the official Cityscapes benchmark to evaluate performance. The original test cannot be used to evaluate model accuracy locally. 

Use some data from the original train split to create test split, but exclude it.

In [ ]:
# Train directory
gtFineDIR_train = os.path.join(LABELS_DIR, 'train')
ImgDIR_train = os.path.join(IMAGES_DIR, 'train')

dataset_train = os.path.join(DATASET_DIR,'train')

# Validation directory
gtFineDIR_val = os.path.join(LABELS_DIR, 'val')
ImgDIR_val = os.path.join(IMAGES_DIR, 'val')

dataset_valid = os.path.join(DATASET_DIR,'val')

# Test directory
dataset_test = os.path.join(DATASET_DIR, 'test')

In [ ]:
# copy images and labels per cities into one target directory
def create_splits(sourcedir, targetdir, cities_list):
    extensions = ['labelIds', 'leftImg8bit'] # only labels and images needed
    os.makedirs(targetdir, exist_ok=True)
    
    for city in tqdm(cities_list, desc="Cities"):
        directory_city = os.path.join(sourcedir, city)
        images = os.listdir(directory_city)
        for image in images:
            if any(word in image for word in extensions):
                image_path = os.path.join(directory_city, image)
                try:
                    shutil.copy(image_path, targetdir)
                except OSError as e:
                    tqdm.write(str(e))

Keep entire cities per split instead of randomly selecting images from all cities. This is to prevent data leakage from similar looking frames and to improve generalization.

In [ ]:
train_cities = ['aachen', 'bochum', 'bremen', 'cologne', 'darmstadt', 'erfurt', 'hamburg', 'hanover', 'krefeld', 'monchengladbach', 'strasbourg', 'tubingen', 'ulm', 'weimar', 'zurich']

create_splits(gtFineDIR_train, dataset_train, train_cities)
create_splits(ImgDIR_train, dataset_train, train_cities)

In [ ]:
valid_cities = ['frankfurt', 'lindau', 'munster']

create_splits(gtFineDIR_val, dataset_valid, valid_cities)
create_splits(ImgDIR_val, dataset_valid, valid_cities)

In [ ]:
test_cities = ['dusseldorf', 'jena', 'stuttgart']

create_splits(gtFineDIR_train, dataset_test, test_cities)
create_splits(ImgDIR_train, dataset_test, test_cities)

In [ ]:
# Roughly 70 / 15 / 15 split
for folder in os.listdir(DATASET_DIR):
    path = os.path.join(DATASET_DIR, folder)
    print(folder, len(os.listdir(path))//2) # divide by 2 because each image has a matching mask file

# Exploratory Dataset Analysis

### Defining functions for visualisation

In [ ]:
# function to save figures and plots
def save_figure(filename, dpi=300):
    path = FIGURE_DIR / filename

    plt.savefig(
        path,
        dpi=dpi,
        bbox_inches="tight",
        facecolor="white"
    )

    print(f"Saved: {path}")

# converts the masks with the original label IDs to masks with the train IDs (0-18) and ignores the rest (255)
def convert_to_train_ids(mask):
    mask = np.asarray(mask)

    if mask.ndim != 2:
        raise ValueError("Expected a 2D labelIds mask")
    if not np.issubdtype(mask.dtype, np.integer):
        raise ValueError("Mask must contain integer label IDs")

    result = np.full(mask.shape, IGNORE_INDEX, dtype=np.uint8)
    valid = (mask >= 0) & (mask < 256)
    result[valid] = ID_LOOKUP[mask[valid].astype(np.int64)]

    return result

# add colour to the greyscale masks for better visualisation
def colorize_mask(train_mask):
    h, w = train_mask.shape
    colored = np.zeros((h, w, 3), dtype=np.uint8)

    for class_id, color in enumerate(CLASS_COLOURS):
        colored[train_mask == class_id] = color

    return colored

# display a sample image, its corresponding segmentation mask, and an overlay of the two
def visualize_sample(samples, fig_title):
    num_samples = len(samples)
    fig, axes = plt.subplots(
        num_samples, 3, 
        figsize=(12, 2 * num_samples), 
        gridspec_kw={'wspace': 0.05, 'hspace': 0.08}
    )

    for i, sample in enumerate(samples):
        # process images and masks
        image = np.asarray(Image.open(sample["image_path"]).convert("RGB"))
        original_mask = np.asarray(Image.open(sample["mask_path"]))
        train_mask = convert_to_train_ids(original_mask)
        color_mask = colorize_mask(train_mask)

        # column 0: Original RGB Image
        axes[i, 0].imshow(image)
        if i == 0:
            axes[i, 0].set_title("Original RGB Image", fontsize=14)
        axes[i, 0].set_ylabel(f"Split: {sample['split']} | City: {sample['city']}", fontsize=6, fontweight="bold")

        # column 1: Segmentation Mask
        axes[i, 1].imshow(color_mask)
        if i == 0:
            axes[i, 1].set_title("19-Class Segmentation Mask", fontsize=14)

        # column 2: Overlay
        axes[i, 2].imshow(image)
        axes[i, 2].imshow(color_mask, alpha=0.5)
        if i == 0:
            axes[i, 2].set_title("Segmentation Overlay", fontsize=14)

    # turn off axis ticks for all subplots
    for row in axes:
        for ax in row:
            ax.set_xticks([])
            ax.set_yticks([])

    save_figure(fig_title)
    plt.show()

### Visualising the dataset

In [ ]:
# create a dataframe with each image path and its corresponding mask path
records = []

for split in SPLITS:
    split_dir = Path(DATASET_DIR) / split

    image_paths = sorted(split_dir.glob("*_leftImg8bit.png"))

    for image_path in image_paths:
        base_name = image_path.name.replace("_leftImg8bit.png", "")
        mask_path = split_dir / (base_name + "_gtFine_labelIds.png")
        city = base_name.split("_")[0]

        records.append({
            "split": split,
            "city": city,
            "image_name": image_path.name,
            "image_path": str(image_path),
            "mask_path": str(mask_path),
            "mask_exists": mask_path.exists()
        })

df = pd.DataFrame(records)

display(df.head())

In [ ]:
# making sure no images are missing their masks
missing = df[~df["mask_exists"]]
display(missing)

for split in SPLITS:
    split_dir = Path(DATASET_DIR) / split
    images = list(split_dir.glob("*_leftImg8bit.png"))
    masks = list(split_dir.glob("*_gtFine_labelIds.png"))
    print(split, "| Images:", len(images), "| Masks:", len(masks))

All images have a corresponding mask.

In [ ]:
# calculate training/validation/test split counts and percentages
split_counts = (
    df.groupby("split")
      .size()
      .reindex(SPLITS, fill_value=0)
)

split_percentages = (100 * split_counts / split_counts.sum())

split_summary = pd.DataFrame({
    "Images": split_counts,
    "Percentage": split_percentages.round(2)
})

display(split_summary)

In [ ]:
# plot the distribution of images across splits
plt.figure(figsize=(8, 5))
sns.barplot(x=split_counts.index, y=split_counts.values)

plt.title("Cityscapes Dataset Split Distribution")
plt.xlabel("Dataset Split")
plt.ylabel("Number of Images")
plt.tight_layout()
save_figure("dataset_split_distribution.png")
plt.show()

In [ ]:
# display the number of images per city for each split
city_counts = (df.groupby(["split", "city"]).size().reset_index(name="image_count"))

plt.figure(figsize=(14, 6))
sns.barplot(
    data=city_counts,
    x="city",
    y="image_count",
    hue="split"
)

plt.title("Number of Images per City")
plt.xlabel("City")
plt.ylabel("Number of Images")
plt.xticks(rotation=65, ha="right")
plt.tight_layout()
save_figure("num_images_per_city.png")
plt.show()

city_stats = (
    city_counts["image_count"]
    .agg(
        city_count="count",
        total_images="sum",
        mean="mean",
        std="std",
        minimum="min",
        q1=lambda x: x.quantile(0.25),
        median="median",
        q3=lambda x: x.quantile(0.75),
        maximum="max"
    )
    .round(2)
)

display(city_stats)

Cities like Strasbourg and Bremen have significantly more images than cities like Bochum and Darmstadt. This means that the cities with more images will have a greater influence on the model's learning. This could matter if the city with more images has a very different street and road structure than other cities.

In [ ]:
# check for overlapping cities between splits to identify any data leakage
split_cities = {
    split: set(df.loc[df["split"] == split, "city"])
    for split in SPLITS
}

for i, split_a in enumerate(SPLITS):
    for split_b in SPLITS[i + 1:]:
        overlap = split_cities[split_a] & split_cities[split_b]

        print(
            f"{split_a} vs {split_b}:",
            overlap if overlap else "No city overlap"
        )

There is no overlap of images between different cities, so choosing to split our training, validation, and test sets by city ensures no data leakage. If there was an overlap between training and test cities, the model may remember an image in the test set from the training and apply the previous training mask rather than identifying and segmenting the image from scratch.

In [ ]:
# gather image metadata such as dimensions and colour mode for both images and masks
# allows us to make sure all images are the same size and have the same number of input channels (3 for RGB)
metadata = []

for row in tqdm(df.itertuples(index=False), total=len(df)):
    if not row.mask_exists:
        continue

    try:
        with Image.open(row.image_path) as image:
            width, height = image.size
            image_mode = image.mode

        with Image.open(row.mask_path) as mask:
            mask_width, mask_height = mask.size
            mask_mode = mask.mode

        metadata.append({
            "split": row.split,
            "city": row.city,
            "image_path": row.image_path,
            "mask_path": row.mask_path,
            "width": width,
            "height": height,
            "image_mode": image_mode,
            "mask_mode": mask_mode,
            "dimensions_match": (width == mask_width and height == mask_height)
        })

    except Exception as e:
        print("Error:", row.image_path, e)

meta_df = pd.DataFrame(metadata)
display(meta_df.head())
print(meta_df[["width", "height"]].describe())
print("Dimension mismatches:", (~meta_df["dimensions_match"]).sum())

All images and masks have the same resolution (2048x1024) which means we do not have to apply any cropping or resizing as part of out pre-processing. We will keep the original resolution and maybe crop or resize during training depending on the training speed and need for data augmentation.

In [ ]:
# select a random image from each split and display it beside its annotated mask and overlay
sample_list = [
    df[(df["split"] == split) & (df["mask_exists"])].sample(1, random_state=42).iloc[0]
    for split in SPLITS
]

visualize_sample(sample_list, "combined_splits_visualization.png")

All images and masks align correctly and the segmentation is accurate. A slight setback is that multiple vehicles or vegetation that are close together in the image are segmented as one group in the mask. The labels appear consistent across the three samples, i.e. cars are blue, vegetation is green, humans are red, etc.

In [ ]:
# inspect the pixel labels across the masks and compare against the 19 chosen classes
raw_label_counts = Counter()

for mask_path in tqdm(meta_df["mask_path"]):
    mask = np.asarray(Image.open(mask_path))
    labels, counts = np.unique(mask, return_counts=True)

    raw_label_counts.update({
        int(label): int(count)
        for label, count in zip(labels, counts)
    })

raw_label_df = pd.DataFrame([
    {"label_id": k, "pixel_count": v}
    for k, v in raw_label_counts.items()
])

raw_label_df["train_id"] = (raw_label_df["label_id"].map(LABEL_TO_TRAIN).fillna(IGNORE_INDEX).astype(int))
raw_label_df["status"] = np.where(raw_label_df["train_id"] == IGNORE_INDEX, "IGNORED", "Training class")

raw_label_df["class_name"] = raw_label_df["train_id"].apply(
    lambda x: CLASS_NAMES[x]
    if x != IGNORE_INDEX else "IGNORED"
)

display(raw_label_df.sort_values("pixel_count", ascending=False))

We see the CityScapes dataset's original 34 classes, of which we use 19 and map the rest to 255 (IGNORED). As expected, road, buildings, and vegetation cover the largest number of pixels across all images in the dataset. Buses, trains, traffic lights, riders, and motorcycles are among the smallest.

In [ ]:
# analyse class distribution and check for imbalance
pixel_records = []
image_class_records = []

for row in tqdm(meta_df.itertuples(index=False), total=len(meta_df)):
    original_mask = np.asarray(Image.open(row.mask_path))
    train_mask = convert_to_train_ids(original_mask)
    counts = np.bincount(train_mask.ravel(), minlength=256)
    total_pixels = train_mask.size

    record = {
        "split": row.split,
        "city": row.city,
        "image_path": row.image_path,
        "total_pixels": total_pixels,
        "ignored_pixels": int(counts[255])
    }

    for class_id, class_name in enumerate(CLASS_NAMES):
        count = int(counts[class_id])
        record[class_name] = count

        image_class_records.append({
            "split": row.split,
            "city": row.city,
            "image_path": row.image_path,
            "class_name": class_name,
            "pixel_count": count,
            "present": count > 0,
            "area_percentage": 100 * count / total_pixels
        })

    pixel_records.append(record)

pixel_df = pd.DataFrame(pixel_records)  # one row per image with pixel counts for each class
image_class_df = pd.DataFrame(image_class_records) # one row per image/class combination with class presence and area percentage

In [ ]:
# global class distribution across the entire dataset
# analyses the amount of space each class occupies in the dataset (how many pixels it covers)
class_totals = (
    pixel_df[CLASS_NAMES]
    .sum()
    .sort_values(ascending=False)
)

total_valid_pixels = class_totals.sum()
class_percentages = (
    class_totals / total_valid_pixels * 100
)

order = class_percentages.index

plt.figure(figsize=(14, 6))
sns.barplot(
    x=order,
    y=class_percentages.values,
    hue=order,
    palette=CLASS_COLOURS_HEX,
    legend=False
)

plt.title("Pixel Distribution Across 19 Cityscapes Classes")
plt.xlabel("Semantic Class")
plt.ylabel("Percentage of Valid Pixels")
plt.xticks(rotation=65, ha="right")
plt.tight_layout()
save_figure("pixel_distribution_by_class.png")
plt.show()

Very severe class imbalance, which was expected. Over 35% of the pixels across all images in the dataset contain road, then about 23% take up buildings, and about 16% make up vegetation. Classes like bus, train, traffic light, rider, and motorcycle take up a very small amount of pixels, seemingly less than 1% each. This imbalance may bias the model's optimisation towards the more densely represented classes.

**Solution:** Weighted loss functions such as weighted cross-entropy.

In [ ]:
# image-level class frequency distribution (how many images contain each class)
class_presence = (
    image_class_df
    .groupby("class_name")["present"]
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

plt.figure(figsize=(14, 6))
sns.barplot(
    x=class_presence.index,
    y=class_presence.values,
    hue=class_presence.index,
    palette=CLASS_COLOURS_HEX,
    legend=False
)

plt.title("Percentage of Images Containing Each Class")
plt.ylabel("Images Containing Class (%)")
plt.xlabel("Class")
plt.xticks(rotation=65, ha="right")
plt.tight_layout()
save_figure("image-level_class_distribution.png")
plt.show()

Despite the severe imbalance in classes, each class does appear in at least 1% of the total images in the dataset. Common features such as poles, buildings, roads, vegetation, cars, traffic signs, and sidewalks appear in over 90% of images, while motorcycles, trucks, buses, adn trains appear in less than 20%.

For classes that appear frequently but occupy a small number of pixels, it's recommended to try to preserve the resolution.  
For classes that are both rare and take up little space, we should consider targeted sampling, but avoid oversampling so that the model doesn't memorise a small number of examples.

In [ ]:
# class distribution across training, validation, and test splits
split_class_counts = (
    pixel_df.groupby("split")[CLASS_NAMES]
    .sum()
    .reindex(SPLITS)
)

split_class_percentages = (
    split_class_counts.div(
        split_class_counts.sum(axis=1),
        axis=0
    ) * 100
)

plt.figure(figsize=(16, 5))
sns.heatmap(
    split_class_percentages,
    annot=True,
    fmt=".2f",
    cmap="YlGnBu"
)

plt.title("Class Pixel Distribution Across Dataset Splits")
plt.xlabel("Class")
plt.ylabel("Split")
plt.tight_layout()
save_figure("class_distribution_by_split.png")
plt.show()

The class distribution between splits is very uniform, almost identical, so the splits are well-representative of each class. No classes appear much more often in training than in validation or testing, so the model should not derive any bias from our choice of split.

In [ ]:
# class distribution by city
city_class_counts = (
    pixel_df.groupby(["split", "city"])[CLASS_NAMES]
    .sum()
)

city_class_percentages = (
    city_class_counts.div(
        city_class_counts.sum(axis=1),
        axis=0
    ) * 100
)

plt.figure(figsize=(18, 10))

sns.heatmap(
    city_class_percentages,
    cmap="viridis",
    linewidths=0.3
)

plt.title("Class Distribution Across Cityscapes Cities")
plt.xlabel("Semantic Class")
plt.ylabel("Dataset Split / City")
plt.tight_layout()
save_figure("class_distribution_by_city.png")
plt.show()

The class distribution between cities is also very uniform, meaning it's good and balanced to keep our splits separated by city. No one city has a much denser presence of any particular class than another.

In [ ]:
# class diversity by city
plt.figure(figsize=(15, 6))

pixel_df["num_classes_present"] = (pixel_df[CLASS_NAMES] > 0).sum(axis=1)

sns.boxplot(
    data=pixel_df,
    x="city",
    y="num_classes_present",
    hue="split"
)

plt.title("Class Diversity per Image Across Cities")
plt.xticks(rotation=65, ha="right")
plt.tight_layout()
save_figure("class_diversity_by_city.png")
plt.show()

Most cities have similar image-level class diversity, but Lindau stands out with significantly lower class diversity. From previous visuals, we saw that Lindau contains the least amount of images out of each city in the dataset, although this doesn't necessarily explain the lower number of classes per image. This is generally a sign of a different scene composition and a different type of urban environment than what is typically seen in the other cities. We decided to keep it in our split to once again preserve the range of visual conditions represented in the dataset.

In [ ]:
# identify rare classes and which images contain them
# 3 rarest classes (truck, bus, train)
rare_classes = class_presence[len(class_presence)-3:].index.tolist()

for class_name in rare_classes:
    subset = image_class_df[
        (image_class_df["class_name"] == class_name) &
        (image_class_df["present"])
    ]

    print(class_name, "| Images containing class:", len(subset))

    display(
        subset.groupby(["split", "city"])
        .size()
        .rename("images")
        .reset_index()
    )

We took the three rarest classes (bus, truck, and train) and analysed how many images each of them appeared in. Even the rarest class was present in over 150 images.

In [ ]:
# analyse image brightness and RGB distributions
rgb_records = []

for row in tqdm(meta_df.itertuples(index=False),
                total=len(meta_df)):

    with Image.open(row.image_path) as im:
        image = np.asarray(
            im.convert("RGB").resize((256, 128))
        )

    rgb_records.append({
        "split": row.split,
        "city": row.city,
        "mean_red": image[:, :, 0].mean(),
        "mean_green": image[:, :, 1].mean(),
        "mean_blue": image[:, :, 2].mean(),
        "brightness": image.mean(),
        "contrast": image.std()
    })

rgb_df = pd.DataFrame(rgb_records)

In [ ]:
# plot brightness by city
plt.figure(figsize=(15, 6))

sns.boxplot(
    data=rgb_df,
    x="city",
    y="brightness",
    hue="split"
)

plt.title("Image Brightness Across Cityscapes Cities")
plt.xticks(rotation=65, ha="right")
plt.tight_layout()
save_figure("image_brightness_by_city.png")
plt.show()

There is some expected variance between image brightness in different cities, which may be down to the weather or time of day when the image was taken, and also due to the colour of the architecture in different cities. The variance is not excessive and there are no major outliers, so we kept the data as is. We could therefore use standard RBG normalisation.

In [ ]:
# plot contrast by city
plt.figure(figsize=(15, 6))

sns.boxplot(
    data=rgb_df,
    x="city",
    y="contrast",
    hue="split"
)

plt.title("Image Contrast Across Cities")
plt.xticks(rotation=65, ha="right")
plt.tight_layout()
save_figure("image_contrast_by_city.png")
plt.show()

As with image brightness, there is some variance in image contrast between cities, but no major outliers, so standard RBG normalisation would suffice.

In [ ]:
# class co-occurrence analysis (which classes tend to appear together in the same image)
presence_matrix = (
    pixel_df[CLASS_NAMES] > 0
).astype(int)

cooccurrence = (
    presence_matrix.T @ presence_matrix
)

plt.figure(figsize=(13, 11))

sns.heatmap(
    cooccurrence,
    cmap="Blues"
)

plt.title("Semantic Class Co-occurrence Matrix")
plt.xlabel("Class")
plt.ylabel("Class")
plt.tight_layout()
save_figure("class_cooccurrence_matrix.png")
plt.show()

We identified a high co-occurance between classes like road, sidewalk, building, pole, traffic sign, vegeation, car, and sky, which was expected as these objects often appear together in city streetviews and these are among the more common classes in the dataset. We also noticed that the rare classes, namely truck, bus, train, and motorcycle, do not appear frequently with many other classes at all.

In [ ]:
# validate split integrity and check for duplicates
import hashlib

def calculate_hash(path):
    hasher = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            hasher.update(chunk)

    return hasher.hexdigest()

hash_records = []

for row in tqdm(meta_df.itertuples(index=False),
                total=len(meta_df)):

    hash_records.append({
        "split": row.split,
        "city": row.city,
        "image_path": row.image_path,
        "hash": calculate_hash(row.image_path)
    })

hash_df = pd.DataFrame(hash_records)

duplicates = hash_df[
    hash_df.duplicated("hash", keep=False)
]

display(duplicates.sort_values("hash"))

We confirmed once again that there was no image overlap, duplication, or data leakage between our training, validation, and test sets.

In [ ]:
# save tables for training
df.to_csv(TABLES_DIR / "dataset_inventory.csv", index=False)
meta_df.to_csv(TABLES_DIR / "image_metadata.csv", index=False)
pixel_df.to_csv(TABLES_DIR / "pixel_statistics.csv", index=False)
image_class_df.to_csv(TABLES_DIR / "image_class_statistics.csv", index=False)
city_class_percentages.to_csv(TABLES_DIR / "city_class_distribution.csv")
split_class_percentages.to_csv(TABLES_DIR / "split_class_distribution.csv")
rgb_df.to_csv(TABLES_DIR / "rgb_statistics.csv", index=False)

print("EDA results saved.")

# Model and Training

Create helper scripts

In [ ]:
Path("src").mkdir(parents=True, exist_ok=True)

In [ ]:
%%writefile src/dataset.py
import os
import random
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms
from torchvision.transforms import functional as TF
from src.utils import convert_labelids_to_trainids, mask_to_long_tensor

IMAGE_SUFFIX = '_leftImg8bit.png'
LABEL_SUFFIX = '_gtFine_labelIds.png'

image_transform = transforms.Compose([
    transforms.ToTensor() # tensor scaled to [0, 1]
])

label_transform = transforms.Compose([
    transforms.PILToTensor(), # tensor without scaling (stays 0-255)
    # remove dummy channel dimension and cast to int64 because PyTorch's CrossEntropyLoss requires this for segmentation. [1, H, W] -> [H, W]
    transforms.Lambda(mask_to_long_tensor)
])

# random scale then crop to a fixed size
def random_crop_scale(image, mask, size, scale=(0.5, 2.0)):
    factor = random.uniform(*scale)

    # window in the original image that gets resized to the crop size
    height = round(size[0] / factor)
    width = round(size[1] / factor)

    top = random.randint(0, image.height - height)
    left = random.randint(0, image.width - width)

    image = TF.resized_crop(image, top, left, height, width, size, interpolation=TF.InterpolationMode.BILINEAR)
    mask = TF.resized_crop(mask, top, left, height, width, size, interpolation=TF.InterpolationMode.NEAREST)

    return image, mask

# random horizontal flip
def random_flip(image, mask):
    if random.random() < 0.5:
        image = TF.hflip(image)
        mask = TF.hflip(mask)

    return image, mask

# resize whole image to a fixed size
def resize(image, mask, size):
    image = TF.resize(image, size, interpolation=TF.InterpolationMode.BILINEAR)
    mask = TF.resize(mask, size, interpolation=TF.InterpolationMode.NEAREST)

    return image, mask

# Docs: https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html
class CityscapesDataset(Dataset):
    def __init__(self, img_dir, image_transform=image_transform, label_transform=label_transform, crop_size=None, crop_scale=(0.5, 2.0), resize=None, flip=False):
        self.img_dir = img_dir
        self.image_transform = image_transform
        self.label_transform = label_transform
        # for training
        self.crop_size = crop_size
        self.crop_scale = crop_scale
        self.resize = resize
        self.flip = flip

        self.images = sorted(file for file in os.listdir(img_dir) if file.endswith(IMAGE_SUFFIX)) # image files

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.images[idx])
        mask_path = img_path.replace(IMAGE_SUFFIX, LABEL_SUFFIX) # get matching mask

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path)

        # labelId to trainId mapping (19 classes + ignore=255)
        mask = convert_labelids_to_trainids(mask)

        # crop or resize
        if self.crop_size:
            image, mask = random_crop_scale(image, mask, self.crop_size, self.crop_scale)
        elif self.resize:
            image, mask = resize(image, mask, self.resize)

        if self.flip:
            image, mask = random_flip(image, mask)

        if self.image_transform:
            image = self.image_transform(image)
        if self.label_transform:
            mask = self.label_transform(mask)
        
        return image, mask

In [ ]:
%%writefile src/evaluate.py
import torch
from tqdm import tqdm
from cityscapesscripts.helpers.labels import labels

# Based on Cityscapes benchmark evaluation metrics
# Source: https://www.cityscapes-dataset.com/benchmarks/
def metrics_from_confusion_matrix(confusion_matrix, num_classes=19):
    confusion_matrix = confusion_matrix.double()
    true_positives = confusion_matrix.diag()
    ground_truth_pixels_per_class = confusion_matrix.sum(dim=1)
    predicted_pixels_per_class = confusion_matrix.sum(dim=0)

    false_positives = predicted_pixels_per_class - true_positives
    false_negatives = ground_truth_pixels_per_class - true_positives

    iou_denominator_per_class = (
        true_positives + false_positives + false_negatives
    )

    iou_per_class = torch.where(
        iou_denominator_per_class > 0,
        true_positives / iou_denominator_per_class.clamp(min=1),
        float("nan"),
    )

    accuracy_per_class = torch.where(
        ground_truth_pixels_per_class > 0,
        true_positives / ground_truth_pixels_per_class.clamp(min=1),
        float("nan"),
    )

    total_ground_truth_pixels = ground_truth_pixels_per_class.sum()

    pixel_accuracy = (
        true_positives.sum() / total_ground_truth_pixels
        if total_ground_truth_pixels > 0
        else torch.tensor(float("nan"), device=confusion_matrix.device)
    )

    class_names = [label.name
        for label in sorted(labels, key=lambda label: label.trainId)
        if 0 <= label.trainId < num_classes
    ]

    return {
        "pixel_accuracy": pixel_accuracy.item(),
        "mean_iou_class": torch.nanmean(iou_per_class).item(),
        "mean_class_accuracy": torch.nanmean(accuracy_per_class).item(),
        "class_names": class_names,
        "iou_per_class": iou_per_class.cpu().tolist(),
        "accuracy_per_class": accuracy_per_class.cpu().tolist(),
    }

@torch.no_grad()
def evaluate(model, loader, device, num_classes=19, ignore_index=255, criterion=None):
    model.eval()

    total_loss = 0.0

    # confusion matrix over the entire set
    # rows = ground-truth
    # cols = predicted
    confusion_matrix = torch.zeros(num_classes, num_classes, dtype=torch.int64, device=device)

    for images, truth_masks in tqdm(loader, leave=False):
        # move images to device
        images = images.to(device, non_blocking=True)
        truth_masks = truth_masks.to(device, non_blocking=True).long()

        logits = model(images)

        # optional validation loss from the same forward pass
        if criterion is not None:
            total_loss += criterion(logits, truth_masks).item()

        # get predicted class for each pixel
        predicted_masks = logits.argmax(dim=1)

        # exclude ignored classes
        valid_pixels = (
            (truth_masks >= 0)
            & (truth_masks < num_classes)
            & (truth_masks != ignore_index)
        )

        # pair truth/prediction
        encoded_class_pairs = (num_classes * truth_masks[valid_pixels] + predicted_masks[valid_pixels])

        # add batch pixel counts to the confusion matrix
        confusion_matrix += torch.bincount(encoded_class_pairs, minlength=num_classes**2).reshape(num_classes, num_classes)

    # evaluation metrics based on confusion matrix
    metrics = metrics_from_confusion_matrix(confusion_matrix, num_classes)

    if criterion is not None:
        metrics["loss"] = total_loss / len(loader)

    return metrics


In [ ]:
%%writefile src/model.py
import torch
import torch.nn.functional as F

NUM_CLASSES = 19    # number of output channels (19 Cityscapes classes)
BASE = 32           # initial number of feature channels, doubles after each downsampling step

# U-Net implementation based on kerrgarr's model
# Source: https://github.com/kerrgarr/SemanticSegmentationCityscapes/blob/main/seg_code/models.py

# input = [B, 3, 512, 1024]      B = Batch Size, 3 = RGB channels, 512x1024 = image size
# output = [B, 19, 512, 1024]    B = Batch Size, 19 = number of classes, 512x1024 = image size


class DoubleConv(torch.nn.Module):
    """
    Two convolution layers, optionally with a parallel shortcut path
    Benefits of residual=True: network can preserve info, and can optimise gradient flow during backpropagation
    """
    def __init__(self, in_channels, out_channels, residual=False):
        super().__init__()

        # main convolution path to learn two features, almost identical to original implementation
        # exclude bias because it is redundant with BatchNorm after
        self.conv1 = torch.nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.norm1 = torch.nn.BatchNorm2d(out_channels)
        self.conv2 = torch.nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.norm2 = torch.nn.BatchNorm2d(out_channels)
        self.relu = torch.nn.ReLU(inplace=True)

        # optional shortcut path to carry information forward
        # applies a shortcut to convert the input shape to match output shape so that it can be added to the main conv path
        self.residual = residual
        if residual:
            if in_channels != out_channels:
                self.shortcut = torch.nn.Sequential(
                    torch.nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
                    torch.nn.BatchNorm2d(out_channels)
                )
            else:
                # if channels already match, return the input unchanged, no new parameters
                self.shortcut = torch.nn.Identity()

    # the forward method actually executes the layers, it's called automatically when you apply DoubleConv(x)
    def forward(self, x):
        # execute the layers defined above
        out = self.conv1(x)
        out = self.norm1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.norm2(out)

        # add the shortcut representation to the main conv path before the final relu, if residual is enabled
        if self.residual:
            out = out + self.shortcut(x)

        return self.relu(out)


class Down(torch.nn.Module):
    """Encoder downsampling with maxpool to reduce spatial dimensions then double conv to extract features"""

    def __init__(self, in_channels, out_channels, residual=False):
        super().__init__()
        
        self.pool = torch.nn.MaxPool2d(kernel_size=2)  # 2x2 max pooling to reduce spatial dimensions by half
        self.conv = DoubleConv(in_channels, out_channels, residual) # two convolution layers, optionally residual

    def forward(self, x):
        return self.conv(self.pool(x))


class Up(torch.nn.Module):
    """Decoder upsampling with bilinear interpolation then double conv"""

    def __init__(self, in_channels, out_channels, residual=False):
        super().__init__()

        # transposed convolution learns the upsampling weights, has learnable parameters
        self.up = torch.nn.ConvTranspose2d(in_channels , in_channels // 2, kernel_size=2, stride=2)
        self.conv = DoubleConv(in_channels, out_channels, residual)

    def forward(self, x1, x2):
        """x1 = low-res decoder features, x2 = high-res encoder features"""
        x1 = self.up(x1)    # upsampling

        # image tensor input has shape [Batch, Channels, Height, Width]
        # compute differences because pooling and upsampling don't always perfectly reverse each other
        # e.g. 255/2 = 127.5 -> 128, then 128*2 = 256
        diffY = x2.size()[2] - x1.size()[2]     # difference in height
        diffX = x2.size()[3] - x1.size()[3]     # difference in width

        # pad the upsampled decoder features to match the encoder features
        # order = [left, right, top, bottom]
        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2, diffY // 2, diffY - diffY // 2])

        # create skip connection along channel dimension (dim=1)
        # combines high-res encoder features with low-res decoder features
        x = torch.cat([x2, x1], dim=1)

        # feature fusion to learn combined encoder and decoder features
        return self.conv(x)


class OutConv(torch.nn.Module):
    """Final classification layer. NOTE: no softmax because the model outputs raw logits"""
    def __init__(self, in_channels, out_channels):
        super(OutConv, self).__init__()
        # 1x1 convolution to map the feature channels to the 19 output classes
        self.conv = torch.nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        return self.conv(x)


class UNet(torch.nn.Module):
    def __init__(self, in_channels=3, num_classes=NUM_CLASSES, base=BASE):
        super(UNet, self).__init__()
        self.in_channels = in_channels
        self.num_classes = num_classes
        self.base = base
        # example input image size                            # [3, 512, 1024]

        # encoder - reducing spatial dimensions and increasing feature channels
        self.inc = DoubleConv(in_channels, base)              # [32, 512, 1024] RGB image to feature channels
        self.down1 = Down(base,   base*2, residual=True)      # [64, 256, 512]
        self.down2 = Down(base*2, base*4, residual=True)      # [128, 128, 256]
        self.down3 = Down(base*4, base*8, residual=True)      # [256, 64, 128]

        # bottleneck - deepest layer with the smallest spatial dimensions and most feature channels
        self.down4 = Down(base * 8, base * 16, residual=True) # [512, 32, 64]

        # decoder - increasing spatial dimensions and reducing feature channels
        self.up1 = Up(base*16, base*8, residual=True)         # [256, 64, 128]
        self.up2 = Up(base*8,  base*4, residual=True)         # [128, 128, 256]
        self.up3 = Up(base*4,  base*2, residual=True)         # [64, 256, 512]
        self.up4 = Up(base*2,  base)                          # [32, 512, 1024]

        # output layer map feature channels to 19 classes
        self.out = OutConv(base, num_classes)                 # [19, 512, 1024]

    def forward(self, x):
        # encoder
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)

        # bottleneck
        b = self.down4(x4)

        # decoder with skip connections
        x = self.up1(b, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        
        return self.out(x)

In [ ]:
%%writefile src/utils.py
import torch
import numpy as np
from pathlib import Path
from PIL import Image
import imageio.v2 as imageio
from cityscapesscripts.helpers.labels import labels

# Official Cityscapes benchmark uses 19 classes for evaluation, remaining classes are ignored
# labelId to trainId mapping (19 classes + ignore=255)
# Benchmark: https://www.cityscapes-dataset.com/benchmarks/
# Source: https://github.com/mcordts/cityscapesScripts/blob/master/cityscapesscripts/helpers/labels.py

# create lookup table for fast conversion
lookup_table = np.full(256, 255, dtype=np.uint8) # default to 255, uint8 = 0-255
for label in labels:
    if 0 <= label.id < len(lookup_table):
        lookup_table[label.id] = label.trainId

def convert_labelids_to_trainids(label_image: Image.Image) -> Image.Image:
    mask = np.asarray(label_image, dtype=np.uint8) # convert mask image to numpy array for lookup table
    mask = lookup_table[mask] # use lookup table to convert labels
    mask = Image.fromarray(mask) # convert back to image
    return mask

def printlabels():
    for label in labels:
        if label.trainId != 255:
            print(f"{label.id:2} -> {label.trainId:2}: {label.name}")

# fails if in notebook so put here
def mask_to_long_tensor(x):
    return x.squeeze(0).long()

# save png of prediciton
def save_prediction(model, image, epoch, output_dir, device):
    model.eval()

    with torch.no_grad():
        pred = model(image.unsqueeze(0).to(device))
        pred = pred.argmax(1).squeeze().cpu().numpy()

    # colour predictions
    colours = np.zeros((256, 3), dtype=np.uint8)
    for label in labels:
        if label.trainId != 255:
            colours[label.trainId] = label.color

    pred = colours[pred]

    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    Image.fromarray(pred.astype(np.uint8)).save(f"{output_dir}/epoch_{epoch:03d}.png")

def make_gif(output_dir, gif_path):
    files = sorted(Path(output_dir).glob("epoch_*.png"))
    imageio.mimsave(gif_path, [imageio.imread(f) for f in files], duration=0.5, loop=0)

In [ ]:
from src.dataset import CityscapesDataset
from src.evaluate import evaluate
from src.model import UNet

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # cpu or cuda
use_amp = device.type == "cuda" # enable Automatic Mixed Precision (AMP) only when using cuda
scaler = torch.amp.GradScaler("cuda", enabled=use_amp) # gradient scaler to help prevent numerical underflow during mixed precision training
torch.backends.cudnn.benchmark = True # selects efficient algorithms for convolution operations when input sizes remain consistent

print("PyTorch version:", torch.__version__)
print("Device:", device)
if torch.cuda.is_available():
    print("   GPU:", torch.cuda.get_device_name(0))
    print("   CUDA version:", torch.version.cuda)

In [ ]:
train_ds = CityscapesDataset(os.path.join(DATASET_DIR, "train"), crop_size=CROP_SIZE, crop_scale=CROP_SCALE, resize=TRAIN_SIZE, flip=FLIP)
valid_ds = CityscapesDataset(os.path.join(DATASET_DIR, "val"))
test_ds = CityscapesDataset(os.path.join(DATASET_DIR, "test"))

train_loader = DataLoader(
    train_ds,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True, # useful for stochastic gradient descent
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
)

valid_loader = DataLoader(
    valid_ds,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False, # no need
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
)

test_loader = DataLoader(
    test_ds,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
)

In [ ]:
model = UNet(in_channels=3, num_classes=NUM_CLASSES, base=BASE).to(device)

# memory estimate does not account for AMP
summary(model, input_size=(TRAIN_BATCH_SIZE, 3, *INPUT_SIZE))

check data formats

In [ ]:
# grab one bactch to inspect shape
# [B = Batch Size, 3 = RGB, H = Height, W = Width]
imgs, masks = next(iter(train_loader))
print("images:", imgs.shape, imgs.dtype)    # should be [B,3,H,W] float32
print("masks :", masks.shape, masks.dtype)  # should be [B,H,W] int64

# run model with no gradient (reduces memory usage) to inspect output shape
#[B = Batch Size, 19 = Classes, H = Height, W = Width]
with torch.no_grad():
    out = model(imgs.to(next(model.parameters()).device))
print("logits:", out.shape)  # should be [B,19,H,W]

select image for training gif

In [ ]:
from src.utils import save_prediction, make_gif
fixed_image, _ = valid_loader.dataset[26]

train loop:
- training = forward + backward passes over training set
- evaluate = forward pass over validation set (metrics + validation loss)

In [ ]:
# class pixel frequencies from exploratory data analysis
freq = pd.read_csv(EDA_CLASS_DIST, index_col="split").loc["train"] # only train split
# [36.779, 6.392, 22.557, 0.622, 0.899, 1.241, 0.206, 0.549, 16.102, 1.158, 4.108, 1.227, 0.147, 6.735, 0.233, 0.260, 0.237, 0.096, 0.450]
freq = torch.tensor(freq.values, dtype=torch.float32)

# median frequency balancing for class imbalance
weights = (freq.median() / freq) ** WEIGHT_EXPONENT 

ce = nn.CrossEntropyLoss(weight=weights.to(device), ignore_index=IGNORE_INDEX) # focuses on per pixel accuracy, weighted to tackle class imbalance
lovasz = LovaszLoss(mode="multiclass", from_logits=True, ignore_index=IGNORE_INDEX) # focuses on intersection over union

def criterion(logits, masks): # loss function
    # combine cross entropy and lovasz loss
    return CE_COEFF * ce(logits, masks) + LOVASZ_COEFF * lovasz(logits, masks)

optimizer = torch.optim.AdamW(model.parameters(), lr=MAX_LEARNING_RATE, weight_decay=WEIGHT_DECAY)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=MIN_LEARNING_RATE) # update learning rate after each epoch

start_epoch = 0
best_valid = float("inf")
best_miou = float("-inf")
history = []

if MODEL_CHECKPOINT.exists():  # if MODEL_CHECKPOINT is a Path
    checkpoint = torch.load(MODEL_CHECKPOINT)

    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scaler.load_state_dict(checkpoint["scaler_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

    start_epoch = checkpoint["epoch"]
    best_valid = checkpoint["best_val_loss"]
    best_miou = checkpoint["best_miou"]
    history = checkpoint.get("history", [])

    print(f"Resuming from epoch {start_epoch + 1}")

def train_epoch(loader):
    model.train()
    total_loss = 0.0

    for imgs, masks in tqdm(loader, leave=False):
        imgs = imgs.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True).long() # int classes required for loss function

        # Automatically use mixed precision to speed up GPU training and reduce VRAM usage
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            logits = model(imgs) # logits = raw predictions
            loss = criterion(logits, masks) # calculate how incorrect predictions

        optimizer.zero_grad() # clear gradients from previous batch
        scaler.scale(loss).backward() # calculate gradients from loss
        scaler.step(optimizer) # update model parameters using calculated gradients
        scaler.update() # help prevent FP16 underflow

        total_loss += loss.item() # add batch loss to total

    scheduler.step() # update learning rate
    
    return total_loss / len(loader)

start_time = time.time()
for epoch in range(start_epoch, EPOCHS):

    train = train_epoch(train_loader)

    # validation loss and metrics in a single forward pass over the validation set
    metrics = evaluate(model, valid_loader, device, num_classes=NUM_CLASSES, ignore_index=IGNORE_INDEX, criterion=criterion)
    valid = metrics["loss"]

    save_prediction(model, fixed_image, epoch + 1, MODEL_GIF, device) # save png

    row = {
        "epoch": epoch + 1,
        "train_loss": train,
        "val_loss": valid,
        "pixel_accuracy": metrics["pixel_accuracy"],
        "mean_iou": metrics["mean_iou_class"],
        "mean_class_accuracy": metrics["mean_class_accuracy"],
    }

    # save per class metrics
    for name, iou, accuracy in zip(metrics["class_names"], metrics["iou_per_class"], metrics["accuracy_per_class"]):
        column = name.replace(" ", "_") # for column names
        row[f"iou_{column}"] = iou
        row[f"accuracy_{column}"] = accuracy

    history.append(row)

    # save training history
    with open(MODEL_HISTORY, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=row.keys())
        writer.writeheader()
        writer.writerows(history)

    # save best model weights
    if metrics["mean_iou_class"] > best_miou:
        best_miou = metrics["mean_iou_class"]
        torch.save(model.state_dict(), MODEL)

    # save checkpoint
    torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_val_loss": best_valid,
            "best_miou": best_miou,
            "history": history,
        },MODEL_CHECKPOINT,
    )

    # calculate estimated time left
    elapsed = time.time() - start_time
    avg_epoch_time = elapsed / (epoch + 1 - start_epoch)
    remaining = avg_epoch_time * (EPOCHS - epoch - 1)

    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} | "
        f"train loss: {train:.4f} | "
        f"valid loss: {valid:.4f} | "
        f"mIoU: {metrics['mean_iou_class']:.4f} | "
        f"pixel acc: {metrics['pixel_accuracy']:.4f} | "
        f"class acc: {metrics['mean_class_accuracy']:.4f} | "
        f"ETA: {time.strftime('%H:%M:%S', time.gmtime(remaining))}"
    )

print(f"\nBest mIoU: {best_miou:.4f}")
print(f"Model: {MODEL}")
print(f"Checkpoint: {MODEL_CHECKPOINT}")
print(f"History: {MODEL_HISTORY}")

In [ ]:
make_gif(MODEL_GIF, MODEL_GIF/"training.gif")

In [ ]:
df = pd.read_csv(MODEL_HISTORY)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Loss curves
axes[0].plot(df["epoch"], df["train_loss"], label="Train Loss")
axes[0].plot(df["epoch"], df["val_loss"], label="Validation Loss")
axes[0].set_title("Cross-Entropy + Lovasz Softmax Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_ylim(0, 3)
axes[0].legend()
axes[0].grid(True)

# Segmentation metrics
axes[1].plot(df["epoch"], df["mean_iou"], label="Mean IoU")
axes[1].plot(df["epoch"], df["pixel_accuracy"], label="Pixel Accuracy")
axes[1].plot(df["epoch"], df["mean_class_accuracy"], label="Mean Class Accuracy")
axes[1].set_title("Validation Metrics")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Score")
axes[1].set_ylim(0, 1)
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig(MODEL_LOSS, dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
df = pd.read_csv(MODEL_HISTORY)

# official class names and colours for trainIds (0-18), sorted by trainId
classes = sorted(
    [l for l in labels if 0 <= l.trainId < 19],
    key=lambda l: l.trainId
)
colors = np.array([l.color for l in classes]) / 255 # matplotlib expects float RGB values [0, 1]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Per-class IoU
for label, color in zip(classes, colors):
    column = label.name.replace(" ", "_") # "traffic light" -> "traffic_light"
    axes[0].plot(df["epoch"], df[f"iou_{column}"], label=label.name, color=color)

axes[0].set_title("Per-Class IoU")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("IoU")
axes[0].set_ylim(0, 1)
axes[0].grid(True)

# Per-class accuracy
for label, color in zip(classes, colors):
    column = label.name.replace(" ", "_")
    axes[1].plot(df["epoch"], df[f"accuracy_{column}"], label=label.name, color=color)
    
axes[1].set_title("Per-Class Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0, 1)
axes[1].grid(True)

# single shared legend because both charts use the same colours
handles, names = axes[0].get_legend_handles_labels()
fig.legend(handles, names, loc="center left", bbox_to_anchor=(1, 0.5), title="Classes")

plt.tight_layout(rect=[0, 0, 1, 1])
plt.savefig(MODEL_CLASSES, dpi=300, bbox_inches="tight")
plt.show()

# Results Evaluation

Load model saved with best weights

In [ ]:
model.load_state_dict(torch.load(MODEL))
model.eval()

Generate sample output from test set

In [ ]:
# Official colours and names for trainIds
# Source: https://github.com/mcordts/cityscapesScripts/blob/master/cityscapesscripts/helpers/labels.py

# keep only trainIds (0-18), sort by trainId
classes = sorted(
    [l for l in labels if 0 <= l.trainId < 19],
    key=lambda l: l.trainId
)

# matplotlib expects float RGB values [0, 1], so divide by 255 to convert
colors = np.array([l.color for l in classes]) / 255
# create colour map with 19 colours + black for unlabelled
cmap = ListedColormap(colors)
cmap.set_over("black")

# get a test image and prediction
imgs, masks = next(iter(test_loader))
model.eval()
with torch.no_grad():
    pred = model(imgs.to(device)).argmax(1).cpu()

# images
img = imgs[0].permute(1, 2, 0).cpu()
true_mask = masks[0].cpu()
pred_mask = pred[0]

# create plot
fig, ax = plt.subplots(2, 1, figsize=(8, 8))

# matplotlib expects (height, width, 3) instead of (3, height, width) so rearrange
ax[0].imshow(img)
ax[0].imshow(true_mask, cmap=cmap, vmin=0, vmax=18, alpha=0.5)
ax[0].set_title("Image with true mask")

ax[1].imshow(pred_mask, cmap=cmap, vmin=0, vmax=18)
ax[1].set_title("Predicted Mask")

# hide plots axis, numbers, borders
for a in ax:
    a.axis("off")

# colour and name + add black for unlabelled
handles = [Patch(color=colors[i], label=l.name) for i, l in enumerate(classes)]
handles.append(Patch(color="black", label="unlabelled"))

fig.legend(handles=handles, loc="center left", bbox_to_anchor=(0.95, 0.5), title="Classes")

plt.tight_layout(rect=[0, 0, 1, 1])
plt.show()

Evaluate on test set to get final results

In [ ]:
# evaluate the model on the test set
results = evaluate(model=model, loader=test_loader, device=device, num_classes=NUM_CLASSES, ignore_index=IGNORE_INDEX)

# build the output
lines = [
    f"Pixel Accuracy:       {results['pixel_accuracy']:.4%}",
    f"Mean IoU (mIoU):      {results['mean_iou_class']:.4%}",
    f"Mean Class Accuracy:  {results['mean_class_accuracy']:.4%}",
    "", "Per-class results:", "-" * 55
]

# per-class results
for name, class_iou, acc in zip(results["class_names"], results["iou_per_class"], results["accuracy_per_class"]):
    iou_str = f"{class_iou:.4%}" if np.isfinite(class_iou) else "N/A"
    acc_str = f"{acc:.4%}" if np.isfinite(acc) else "N/A"

    lines.append(
        f"{name:20s} IoU: {iou_str:>10s} | "
        f"Accuracy: {acc_str:>10s}"
    )

# combine all lines into one string
output = "\n".join(lines)
print(output)

# save results to a text file
with open(MODEL_EVAL, "w", encoding="utf-8") as f:
    f.write(output)

print(f"\nResults saved to {MODEL_EVAL}")

# Cross Fold Validation

folds spilt by city, to avoid near identical frames being in both train and validation 

In [ ]:
del model, optimizer, scheduler, scaler, train_loader, valid_loader
torch.cuda.empty_cache()

In [ ]:
# 5 folds for cross-validation, built from the 15 train + 3 val cities (2,939 images).
# The 3 test cities (dusseldorf, jena, stuttgart) are left out so they stay unseen.
# Whole cities are kept in one fold because frames from the same city look very similar;
# splitting a city across train and validation would leak data and inflate the scores.
# Cities are grouped so each fold has a similar number of images .

FOLDS = {
    1: ["darmstadt", "strasbourg", "zurich"],
    2: ["hamburg", "hanover", "ulm", "lindau"],
    3: ["cologne", "weimar", "frankfurt"],
    4: ["aachen", "bremen", "erfurt"],
    5: ["bochum", "krefeld", "monchengladbach", "tubingen", "munster"],
}

all_cities = []
for fold, cities in FOLDS.items():
    print(f"Fold {fold}: {len(cities)} cities")
    all_cities.extend(cities)

print("Total cities:", len(all_cities))
print("Unique cities:", len(set(all_cities)))

In [ ]:
SOURCE_DIRS = [Path(DATASET_DIR) / "train", Path(DATASET_DIR) / "val"]
CV_DIR = Path("../data/cv")

for fold, val_cities in FOLDS.items():
    fold_dir = CV_DIR / f"fold_{fold}"
    (fold_dir / "train").mkdir(parents=True, exist_ok=True)
    (fold_dir / "val").mkdir(parents=True, exist_ok=True)

    for source_dir in SOURCE_DIRS:
        for file in os.listdir(source_dir):
            city = file.split("_")[0]
            split = "val" if city in val_cities else "train"
            destination = fold_dir / split / file
            if not destination.exists():
                os.link(source_dir / file, destination)

print("Fold folders created")

In [ ]:
for fold in FOLDS:
    for split in ["train", "val"]:
        count = len(list((CV_DIR / f"fold_{fold}" / split).glob("*_leftImg8bit.png")))
        print(f"Fold {fold} {split}: {count}")

Model train image resolution size downsized

In [ ]:
VERSION = "cv"


NATIVE_SIZE = (1024, 2048) # original resolution
CROP_SIZE = None    # random crop size | None = no crops
CROP_SCALE = (0.5, 2.0)    # random scale factor range for crops
TRAIN_SIZE = (256,512)          # resize if no crops | None = native size
FLIP = True                # random horizontal flip

INPUT_SIZE = CROP_SIZE or TRAIN_SIZE or NATIVE_SIZE # training samples size

TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 8
#NUM_WORKERS = min(12, os.cpu_count() or 1)
NUM_WORKERS = 4
print("Dataloader workers:", NUM_WORKERS)

# model
BASE = 32
# criterion
CE_COEFF = 1.0 # weight cross entropy loss
LOVASZ_COEFF = 0.5 # weight lovasz loss
WEIGHT_EXPONENT = 0.5 # class imbalance weights: 0 = no weights, 0.5 = sqrt
# optimizer
MAX_LEARNING_RATE = 1e-4 # start at this rate
MIN_LEARNING_RATE = 1e-6 # slowly reduce to this rate
WEIGHT_DECAY = 1e-4
# training
EPOCHS = 40

In [ ]:
VERSION_SIZE = f"{VERSION}-{INPUT_SIZE[0]}-{INPUT_SIZE[1]}"

MODEL_DIR = Path(MODELS_DIR)/VERSION_SIZE
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(INPUT_SIZE, MODEL_DIR)

In [ ]:
def make_loaders(fold_dir):
    train_ds = CityscapesDataset(fold_dir / "train", crop_size=CROP_SIZE, crop_scale=CROP_SCALE, resize=TRAIN_SIZE, flip=FLIP)
    valid_ds = CityscapesDataset(fold_dir / "val", resize=TRAIN_SIZE)

    train_loader = DataLoader(
        train_ds,
        batch_size=TRAIN_BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=(device.type == "cuda"),
        persistent_workers=(NUM_WORKERS > 0),
    )

    valid_loader = DataLoader(
        valid_ds,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(device.type == "cuda"),
        persistent_workers=(NUM_WORKERS > 0),
    )

    return train_loader, valid_loader


In [ ]:
model = UNet(in_channels=3, num_classes=NUM_CLASSES, base=BASE).to(device)

# memory estimate does not account for AMP
from torchinfo import summary
summary(model, input_size=(TRAIN_BATCH_SIZE, 3, *INPUT_SIZE))

In [ ]:
# grab one bactch to inspect shape
# [B = Batch Size, 3 = RGB, H = Height, W = Width]
train_loader, valid_loader = make_loaders(CV_DIR / "fold_1")
print("train images:", len(train_loader.dataset), "| val images:", len(valid_loader.dataset))
imgs, masks = next(iter(train_loader))
print("images:", imgs.shape, imgs.dtype)    # should be [B,3,H,W] float32
print("masks :", masks.shape, masks.dtype)  # should be [B,H,W] int64

# run model with no gradient (reduces memory usage) to inspect output shape
# [B = Batch Size, 19 = Classes, H = Height, W = Width]
# with torch.no_grad():
#     out = model(imgs.to(next(model.parameters()).device))
# print("logits:", out.shape)  # should be [B,19,H,W]

In [ ]:
# ---------- 1. loss function (same for every fold) ----------
freq = pd.read_csv(EDA_CLASS_DIST, index_col="split").loc["train"]
freq = torch.tensor(freq.values, dtype=torch.float32)
weights = (freq.median() / freq) ** WEIGHT_EXPONENT

ce = nn.CrossEntropyLoss(weight=weights.to(device), ignore_index=IGNORE_INDEX)
lovasz = LovaszLoss(mode="multiclass", from_logits=True, ignore_index=IGNORE_INDEX)

def criterion(logits, masks):
    return CE_COEFF * ce(logits, masks) + LOVASZ_COEFF * lovasz(logits, masks)

# ---------- 2. one training epoch ----------
def train_epoch(model, loader, optimizer, scheduler, scaler):
    model.train()
    total_loss = 0.0
    for imgs, masks in tqdm(loader, leave=False):
        imgs = imgs.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True).long()

        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            logits = model(imgs)
            loss = criterion(logits, masks)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()

    scheduler.step()
    return total_loss / len(loader)

# ---------- 3. the fold loop ----------
results = []

for fold in FOLDS:
    fold_out = MODEL_DIR / f"fold_{fold}"
    fold_out.mkdir(parents=True, exist_ok=True)
    done_file = fold_out / "best.json"

    if done_file.exists():
        print(f"Fold {fold} already finished, skipping")
        with open(done_file) as f:
            results.append(json.load(f))
        continue

    print(f"\n===== Fold {fold}: validating on {FOLDS[fold]} =====")

    torch.manual_seed(42)
    model = UNet(in_channels=3, num_classes=NUM_CLASSES, base=BASE).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=MAX_LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=MIN_LEARNING_RATE)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    train_loader, valid_loader = make_loaders(CV_DIR / f"fold_{fold}")

    history = []
    best = None
    start = time.time()

    for epoch in range(EPOCHS):
        train_loss = train_epoch(model, train_loader, optimizer, scheduler, scaler)
        metrics = evaluate(model, valid_loader, device, num_classes=NUM_CLASSES, ignore_index=IGNORE_INDEX, criterion=criterion)

        row = {
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": float(metrics["loss"]),
            "pixel_accuracy": float(metrics["pixel_accuracy"]),
            "mean_iou": float(metrics["mean_iou_class"]),
            "mean_class_accuracy": float(metrics["mean_class_accuracy"]),
        }
        for name, iou, acc in zip(metrics["class_names"], metrics["iou_per_class"], metrics["accuracy_per_class"]):
            column = name.replace(" ", "_")
            row[f"iou_{column}"] = float(iou)
            row[f"accuracy_{column}"] = float(acc)

        history.append(row)
        pd.DataFrame(history).to_csv(fold_out / "history.csv", index=False)

        if best is None or row["mean_iou"] > best["mean_iou"]:
            best = {"fold": fold, "best_epoch": epoch + 1,
                    "mean_iou": row["mean_iou"],
                    "pixel_accuracy": row["pixel_accuracy"],
                    "mean_class_accuracy": row["mean_class_accuracy"],
                    "val_loss": row["val_loss"]}
            torch.save(model.state_dict(), fold_out / "model.pt")

        print(f"Fold {fold} | Epoch {epoch + 1:02d}/{EPOCHS} | "
              f"train loss: {train_loss:.4f} | val loss: {row['val_loss']:.4f} | "
              f"mIoU: {row['mean_iou']:.4f} | pixel acc: {row['pixel_accuracy']:.4f}")

    best["minutes"] = round((time.time() - start) / 60, 1)
    with open(done_file, "w") as f:
        json.dump(best, f, indent=2)
    results.append(best)

    # free GPU memory before the next fold
    del model, optimizer, scheduler, scaler, train_loader, valid_loader
    torch.cuda.empty_cache()

print("\nAll folds finished")

In [ ]:
rows = []
for fold in FOLDS:
    with open(MODEL_DIR / f"fold_{fold}" / "best.json") as f:
        rows.append(json.load(f))

results_df = pd.DataFrame(rows).set_index("fold")

metrics = ["mean_iou", "pixel_accuracy", "mean_class_accuracy", "val_loss"]
summary = results_df[metrics].agg(["mean", "std"])

table = pd.concat([results_df[["best_epoch"] + metrics + ["minutes"]], summary])
display(table.round(4))

table.to_csv(MODEL_DIR / "cv_results.csv")
print("Saved to", MODEL_DIR / "cv_results.csv")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for fold in FOLDS:
    hist = pd.read_csv(MODEL_DIR / f"fold_{fold}" / "history.csv")
    axes[0].plot(hist["epoch"], hist["mean_iou"], label=f"Fold {fold}")
    axes[1].plot(hist["epoch"], hist["val_loss"], label=f"Fold {fold}")

axes[0].set_title("Validation mIoU per Fold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("mIoU")
axes[0].legend()
axes[0].grid(True)

axes[1].set_title("Validation Loss per Fold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig(MODEL_DIR / "cv_curves.png", dpi=200, bbox_inches="tight")
plt.show()